# z617 - Escalado + Clase (Etapa A, consigna nueva)
Grano cliente-producto, TODOS los productos. `TN(p)` = promedio expanding (todo el historial hasta `p`, inclusive). Target = delta escalado, ambos terminos usan el MISMO `TN(p)` del ancla (nunca mira adelante).

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'CP_ESC01',
    'sellin_zeroes_path': '/home/ds/datasets/sell-in-zeroes.txt',
    'horizonte_meses': 2
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/CP_ESC01


## 1. Cargar (grano customer_id x product_id x periodo, TODOS los productos, ya con ceros)

In [4]:
df = pl.read_csv(PARAM['sellin_zeroes_path'], separator=",").select(
    ["customer_id", "product_id", "periodo", "tn"]
)

CLAVE = ["customer_id", "product_id"]
df = df.sort(CLAVE + ["periodo"])
print(df.shape)

(16648066, 4)


## 2. TN(p): promedio expanding, hacia atras, incluye el periodo actual
`TN(p) = suma acumulada de tn hasta p / cantidad de periodos hasta p`, dentro del par cliente-producto. Los ceros del zero-fill entran de lleno en el promedio (un mes sin compra baja el promedio, es la intencion).

In [5]:
df = df.with_columns([
    pl.col("tn").cum_sum().over(CLAVE).alias("_cumsum_tn"),
    pl.col("tn").cum_count().over(CLAVE).alias("_cumcount"),
])

df = df.with_columns(
    (pl.col("_cumsum_tn") / pl.col("_cumcount")).alias("TN_promedio")
)

df = df.drop(["_cumsum_tn", "_cumcount"])

## 3. Variables de anclaje y clase
`tn0` = tn del periodo ancla (p). `tn1` = tn de p+1 (referencia). `clase_original` = tn de p+2 (target crudo). Todo escalado usa el `TN_promedio` DEL ANCLA (p) -- nunca el de un periodo futuro.

In [6]:
EPS = 1e-6
H = PARAM['horizonte_meses']

df = df.with_columns([
    pl.col("tn").alias("tn0"),
    pl.col("tn").shift(-1).over(CLAVE).alias("tn1"),
    pl.col("tn").shift(-H).over(CLAVE).alias("clase_original"),
])

df = df.with_columns([
    (pl.col("tn0") / (pl.col("TN_promedio") + EPS)).alias("tn0_escalado"),
    (pl.col("clase_original") / (pl.col("TN_promedio") + EPS)).alias("clase_original_escalada"),
])

df = df.with_columns(
    (pl.col("clase_original_escalada") - pl.col("tn0_escalado")).alias("clase")
)

# serie escalada del periodo actual -- base para lags/historia en la proxima etapa
df = df.with_columns(
    pl.col("tn0_escalado").alias("E_tn")
)

## 4. Periodos (para el split train/valid de la proxima etapa)

In [7]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = df.with_columns(
    pl.col("periodo").map_elements(periodo_a_meses, return_dtype=pl.Int64).alias("periodo_m")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

## 5. Verificacion rapida antes de guardar

In [8]:
print(df.shape)
print("nulls en TN_promedio:", df["TN_promedio"].null_count())
print("TN_promedio min/max:", df["TN_promedio"].min(), df["TN_promedio"].max())
print("clase min/max:", df["clase"].min(), df["clase"].max())
df.select(["customer_id", "product_id", "periodo", "TN_promedio", "tn0", "tn1",
           "tn0_escalado", "clase_original", "clase_original_escalada", "clase"]).head(10)

(16648066, 14)
nulls en TN_promedio: 1
TN_promedio min/max: 0.0 243.79554272727273
clase min/max: -33.99948411603997 108605950.00000001


customer_id,product_id,periodo,TN_promedio,tn0,tn1,tn0_escalado,clase_original,clase_original_escalada,clase
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
10001,20001,201701,99.43861,99.43861,198.84365,1.0,92.46537,0.929874,-0.070126
10001,20001,201702,149.14113,198.84365,92.46537,1.333258,13.29728,0.089159,-1.244099
10001,20001,201703,130.24921,92.46537,13.29728,0.709911,101.00563,0.77548,0.065569
10001,20001,201704,101.011228,13.29728,101.00563,0.131642,128.04792,1.26766,1.136019
10001,20001,201705,101.010108,101.00563,128.04792,0.999956,101.20711,1.00195,0.001995
10001,20001,201706,105.51641,128.04792,101.20711,1.213536,43.3393,0.410735,-0.8028
10001,20001,201707,104.900796,101.20711,43.3393,0.964789,289.35024,2.758323,1.793534
10001,20001,201708,97.205609,43.3393,289.35024,0.445852,222.11389,2.28499,1.839139
10001,20001,201709,118.555012,289.35024,222.11389,2.440641,111.54944,0.940909,-1.499732


## 6. Guardar

In [9]:
salida = os.path.join(ruta, "tb_escalado_CP.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/CP_ESC01/tb_escalado_CP.parquet
(16648066, 14)
